# 🚀 Stock Market Prediction System
## Random Forest & Sentiment Analysis Hybrid Model

This notebook orchestrates the entire stock prediction pipeline from data collection to backtesting.

**Pipeline Steps:**
1. Data Collection
2. Data Preprocessing
3. Feature Engineering
4. Model Training & Evaluation
5. Baseline Comparison
6. Backtesting (Walk-Forward Validation)

## 📦 Step 0: Setup and Imports

In [ ]:
import sys
import os

# Add src directory to path so we can import modules
sys.path.append(os.path.join(os.getcwd(), 'src'))

import config
from src.data_collection import (
    download_stock_data,
    download_multiple_stocks,
    load_stock_data
)
from src.preprocessing import preprocess_stock_data
from src.feature_engineering import engineer_all_features, prepare_ml_data
from src.models import train_and_evaluate_pipeline, compare_baseline
from src.backtesting import walk_forward_validation, compare_strategies

print("✅ All imports successful!")

## ⚙️ Configuration

Display current configuration settings:

In [ ]:
# Print configuration
config.print_config()

## 🔧 Function Definitions

### Pipeline for Single Stock

In [ ]:
def run_complete_pipeline(ticker, start_date=None, end_date=None):
    """
    Run the complete ML pipeline for a single stock

    This function orchestrates all steps:
    1. Data Collection
    2. Preprocessing
    3. Feature Engineering
    4. Model Training
    5. Evaluation
    6. Backtesting

    Args:
        ticker (str): Stock ticker symbol (e.g., 'AAPL')
        start_date (str): Start date for data (default from config)
        end_date (str): End date for data (default from config)

    Returns:
        dict: Results from all pipeline steps
    """
    if start_date is None:
        start_date = config.START_DATE
    if end_date is None:
        end_date = config.END_DATE

    print("\\n" + "=" * 80)
    print(" " * 20 + f"STOCK PREDICTION PIPELINE: {ticker}")
    print("=" * 80)
    print(f"Date Range: {start_date} to {end_date}")
    print("=" * 80 + "\\n")

    # ========================================================================
    # STEP 1: DATA COLLECTION
    # ========================================================================
    print("\\n" + "🔵 " * 30)
    print("STEP 1: DATA COLLECTION")
    print("🔵 " * 30)

    # Try to load existing data first
    raw_data = load_stock_data(ticker, 'raw')

    if raw_data is None:
        # Download if not found
        print(f"No existing data found. Downloading {ticker}...")
        raw_data = download_stock_data(ticker, start_date, end_date, save=True)

        if raw_data is None:
            print(f"❌ Failed to download data for {ticker}")
            return None
    else:
        print(f"✅ Loaded existing data for {ticker}")

    print(f"Raw data shape: {raw_data.shape}")

    # ========================================================================
    # STEP 2: DATA PREPROCESSING
    # ========================================================================
    print("\\n" + "🟢 " * 30)
    print("STEP 2: DATA PREPROCESSING")
    print("🟢 " * 30)

    processed_data = preprocess_stock_data(
        raw_data,
        save_path=config.get_data_path(ticker, 'processed')
    )

    print(f"Processed data shape: {processed_data.shape}")

    # ========================================================================
    # STEP 3: FEATURE ENGINEERING
    # ========================================================================
    print("\\n" + "🟡 " * 30)
    print("STEP 3: FEATURE ENGINEERING")
    print("🟡 " * 30)

    feature_data = engineer_all_features(processed_data)

    # Save feature data
    feature_path = config.get_data_path(ticker, 'features')
    feature_data.to_csv(feature_path, index=False)
    print(f"Saved feature data to: {feature_path}")

    # Prepare for ML
    X, y, feature_names = prepare_ml_data(feature_data)

    print(f"\\nFinal dataset:")
    print(f"  Samples: {len(X)}")
    print(f"  Features: {len(feature_names)}")
    print(f"  Target distribution: {y.value_counts().to_dict()}")

    # ========================================================================
    # STEP 4: MODEL TRAINING & EVALUATION
    # ========================================================================
    print("\\n" + "🔴 " * 30)
    print("STEP 4: MODEL TRAINING & EVALUATION")
    print("🔴 " * 30)

    # Run complete training pipeline
    model_results = train_and_evaluate_pipeline(X, y, feature_names)

    # ========================================================================
    # STEP 5: BASELINE COMPARISON
    # ========================================================================
    print("\\n" + "🟣 " * 30)
    print("STEP 5: BASELINE COMPARISON")
    print("🟣 " * 30)

    comparison = compare_baseline(
        model_results['X_train'],
        model_results['y_train'],
        model_results['X_test'],
        model_results['y_test']
    )

    # ========================================================================
    # STEP 6: BACKTESTING
    # ========================================================================
    print("\\n" + "🟠 " * 30)
    print("STEP 6: BACKTESTING (Walk-Forward Validation)")
    print("🟠 " * 30)

    backtest_results = walk_forward_validation(X, y, feature_names,
                                               n_splits=config.N_SPLITS)

    # ========================================================================
    # FINAL SUMMARY
    # ========================================================================
    print("\\n" + "=" * 80)
    print(" " * 25 + "PIPELINE COMPLETE!")
    print("=" * 80)

    print(f"\\n📊 FINAL RESULTS FOR {ticker}:")
    print("-" * 80)
    print(f"Test Set Accuracy:        {model_results['metrics']['test_accuracy']:.2%}")
    print(f"Backtesting Mean Accuracy: {backtest_results['accuracy'].mean():.2%}")
    print(f"Backtesting Std:          {backtest_results['accuracy'].std():.4f}")

    if comparison['improvement'] > 0:
        print(f"\\nSentiment features improved accuracy by: {comparison['improvement']:.2%}")
    else:
        print(f"\\nSentiment features did not improve accuracy")

    print("\\n" + "=" * 80)

    # Return all results
    return {
        'ticker': ticker,
        'raw_data': raw_data,
        'processed_data': processed_data,
        'feature_data': feature_data,
        'X': X,
        'y': y,
        'feature_names': feature_names,
        'model_results': model_results,
        'comparison': comparison,
        'backtest_results': backtest_results
    }

print("✅ Function defined: run_complete_pipeline()")

### Pipeline for Multiple Stocks

In [ ]:
def run_multiple_stocks(tickers=None):
    """
    Run pipeline for multiple stocks

    Args:
        tickers (list): List of ticker symbols (default from config)

    Returns:
        dict: Results for each ticker
    """
    if tickers is None:
        tickers = config.TICKERS

    print("\\n" + "=" * 80)
    print(" " * 15 + f"RUNNING PIPELINE FOR {len(tickers)} STOCKS")
    print("=" * 80)
    print(f"Tickers: {', '.join(tickers)}")
    print("=" * 80 + "\\n")

    results = {}

    for idx, ticker in enumerate(tickers, 1):
        print(f"\\n{'=' * 80}")
        print(f" " * 20 + f"PROCESSING {ticker} ({idx}/{len(tickers)})")
        print(f"{'=' * 80}\\n")

        try:
            ticker_results = run_complete_pipeline(ticker)
            if ticker_results is not None:
                results[ticker] = ticker_results
                print(f"\\n✅ Successfully processed {ticker}")
        except Exception as e:
            print(f"\\n❌ Error processing {ticker}: {e}")
            continue

    # Summary across all stocks
    print("\\n" + "=" * 80)
    print(" " * 20 + "MULTI-STOCK SUMMARY")
    print("=" * 80)

    for ticker, result in results.items():
        test_acc = result['model_results']['metrics']['test_accuracy']
        backtest_acc = result['backtest_results']['accuracy'].mean()
        print(f"{ticker:6s} - Test: {test_acc:.2%}, Backtest: {backtest_acc:.2%}")

    return results

print("✅ Function defined: run_multiple_stocks()")

## ▶️ Execution Section

Choose one of the options below to run the pipeline:

### Option 1: Run Pipeline for Single Stock

Uncomment and modify the ticker as needed

In [ ]:
# Run pipeline for AAPL
# Uncomment the line below to execute

# results_single = run_complete_pipeline('AAPL')

### Option 2: Run Pipeline for Multiple Stocks (from config)

Runs for all stocks defined in `config.TICKERS`

In [ ]:
# Run pipeline for all configured stocks
# Uncomment the line below to execute

# results_multiple = run_multiple_stocks()

### Option 3: Run Pipeline for Custom List of Stocks

In [ ]:
# Run pipeline for custom tickers
# Uncomment and modify the list as needed

# custom_tickers = ['AAPL', 'MSFT', 'GOOGL']
# results_custom = run_multiple_stocks(custom_tickers)

### Option 4: Download Data Only

Download data for all stocks without running the full pipeline

In [ ]:
# Download data for all configured stocks (preparation step)
# Uncomment the line below to execute

# print("\\nDownloading data for all configured stocks...")
# data_dict = download_multiple_stocks(
#     config.TICKERS,
#     config.START_DATE,
#     config.END_DATE,
#     save=True
# )
# print(f"\\n✅ Downloaded data for {len(data_dict)} stocks")

## 📊 Results Analysis

After running one of the pipeline options above, use these cells to analyze results:

### View Single Stock Results

In [ ]:
# Uncomment and run after executing Option 1

# if 'results_single' in locals():
#     print("\\n📈 Model Metrics:")
#     print(results_single['model_results']['metrics'])
#     print("\\n📊 Backtesting Results:")
#     print(f"Mean Accuracy: {results_single['backtest_results']['accuracy'].mean():.2%}")
#     print(f"Std: {results_single['backtest_results']['accuracy'].std():.4f}")
# else:
#     print("No single stock results available. Run Option 1 first.")

### View Multiple Stocks Comparison

In [ ]:
# Uncomment and run after executing Option 2 or 3

# if 'results_multiple' in locals():
#     import pandas as pd
#     
#     summary_data = []
#     for ticker, result in results_multiple.items():
#         summary_data.append({
#             'Ticker': ticker,
#             'Test Accuracy': result['model_results']['metrics']['test_accuracy'],
#             'Backtest Mean': result['backtest_results']['accuracy'].mean(),
#             'Backtest Std': result['backtest_results']['accuracy'].std(),
#             'Features': len(result['feature_names'])
#         })
#     
#     summary_df = pd.DataFrame(summary_data)
#     print("\\n📊 Multi-Stock Summary:")
#     print(summary_df.to_string(index=False))
# else:
#     print("No multiple stock results available. Run Option 2 or 3 first.")

## 📚 Usage Guide

### How to Use This Notebook:

1. **Run Setup Cell**: Execute the imports and configuration cell at the top
2. **Choose an Option**: Uncomment and run one of the execution cells
   - Option 1: Single stock analysis
   - Option 2: All configured stocks
   - Option 3: Custom list of stocks
   - Option 4: Data download only
3. **Wait for Completion**: The pipeline will print progress indicators
4. **Analyze Results**: Use the results analysis cells to view outputs

### Pipeline Overview:

| Step | Description | Output |
|------|-------------|--------|
| 1. Data Collection | Download or load stock data | Raw OHLCV data |
| 2. Preprocessing | Clean and prepare data | Standardized features |
| 3. Feature Engineering | Create ML features | 10+ engineered features |
| 4. Model Training | Train Random Forest models | Trained models + metrics |
| 5. Baseline Comparison | Compare with baseline | Sentiment improvement % |
| 6. Backtesting | Walk-forward validation | Historical accuracy |

### Key Configuration Variables (in `config.py`):

- `TICKERS`: List of stock symbols to analyze
- `START_DATE`, `END_DATE`: Historical date range
- `ROLLING_WINDOWS`: Time windows for moving averages
- `N_SPLITS`: Number of splits for walk-forward validation

### Interpreting Results:

- **Test Accuracy**: Accuracy on held-out test set (70-30 split)
- **Backtest Mean/Std**: Average and standard deviation of accuracy across walk-forward splits
- **Sentiment Improvement**: How much adding sentiment features improved accuracy